# Setup & Load Excel

In [ ]:
import pandas as pd
import numpy as np
import unicodedata

# Import Excel file with each dataset in a separate sheet
xls = pd.ExcelFile("All_Data_DW.xlsx") 

dips = pd.read_excel(xls, "Voltage_Dips")
overs = pd.read_excel(xls, "Overvoltages")
cont = pd.read_excel(xls, "Continuous_Phenomena")
subs = pd.read_excel(xls, "Secondary_Substations")
net = pd.read_excel(xls, "caracteristicas-da-rede")
nuts = pd.read_excel(xls, "nuts_regions")

# Standardize string column names
for df in [dips, overs, cont, subs]:
    if "Installation Code" in df.columns:
        df["Installation Code"] = df["Installation Code"].astype(str).str.strip().str.upper()
    if "Busbar" in df.columns:
        df["Busbar"] = df["Busbar"].astype(str).str.strip().str.upper()

# Glimpse of the datasets
dips.head(), overs.head(), cont.head(), subs.head(), net.head()

(   Year Installation Code Type of installation         Designation  \
 0  2014      0105S5003900           Subestação            Esgueira   
 1  2014      0107S5002500           Subestação             Espinho   
 2  2014      0108S5001000           Subestação              Avanca   
 3  2014      0108S5001000           Subestação              Avanca   
 4  2014      0114S5022700           Subestação  Oliveira do Bairro   
 
           Busbar        Municipality                     NUTS III  \
 0  15 KV - I/III              Aveiro             Região de Aveiro   
 1     15 KV - I1             Espinho  Área Metropolitana do Porto   
 2     15 KV - I1           Estarreja             Região de Aveiro   
 3     15 KV - I2           Estarreja             Região de Aveiro   
 4          15 KV  Oliveira do Bairro             Região de Aveiro   
 
    Nominal Voltage (V) Start Date   End Date  ...  X5  \
 0              15000.0 2013-12-30 2015-01-04  ...   0   
 1              15000.0 2014-01-10

# Create Dimension Tables

In [45]:
# ===========================================
# BUILD DIM_INSTALLATION
# ===========================================

# --- 0. Fix incorrect column names ---
net = net.rename(columns={"Installacion Code": "Installation Code"})
if "Installation Code" in net.columns:
    net["Installation Code"] = net["Installation Code"].astype(str).str.strip().str.upper()

# --- 1. Extract base installation info from event sheets ---
install_cols = ["Installation Code", "Designation", "Type of installation", "Busbar"]

events_inst = pd.concat([
    dips[install_cols],
    overs[install_cols],
    cont[install_cols]
], ignore_index=True)

events_inst = events_inst.drop_duplicates().dropna(subset=["Installation Code"]).reset_index(drop=True)

# --- 2. Merge with BOTH metadata tables ---
# Left join subs (Secondary Substations)
dim_installation = pd.merge(events_inst, subs, on="Installation Code", how="left")

# Left join net (Primary Substations)
dim_installation = pd.merge(dim_installation,  net,  on="Installation Code",  how="left")

# --- 3. Transformation ---

# Split Geographic Coordinates
coords = dim_installation["Geographic Coordinates"].str.split(",", expand=True)

if coords.shape[1] >= 2:
    dim_installation["Latitude"] = coords[0].str.strip()
    dim_installation["Longitude"] = coords[1].str.strip()
else:
    dim_installation["Latitude"] = np.nan
    dim_installation["Longitude"] = np.nan

dim_installation["Latitude"] = pd.to_numeric(dim_installation["Latitude"], errors="coerce")
dim_installation["Longitude"] = pd.to_numeric(dim_installation["Longitude"], errors="coerce")

# Handle improper NaN values
a_cols = ["Contracted Power [kVA]", "Number of Clients", "Power Generation [kW]", "Number of Client Producers"]
for col in a_cols: 
    if col in dim_installation.columns:
        dim_installation[col] = dim_installation[col].replace({"N/D": None, "<20": 19}) 

# Convert numeric columns to proper types
dim_installation["Installed Power [kVA]"]    = pd.to_numeric(dim_installation["Installed Power [kVA]"], errors="coerce")
dim_installation["Contracted Power [kVA]"]   = pd.to_numeric(dim_installation["Contracted Power [kVA]"], errors="coerce")
dim_installation["Power Generation [kW]"]    = pd.to_numeric(dim_installation["Power Generation [kW]"], errors="coerce")
dim_installation["Number of Clients"]        = pd.to_numeric(dim_installation["Number of Clients"], errors="coerce")
dim_installation["Number of Client Producers"] = pd.to_numeric(dim_installation["Number of Client Producers"], errors="coerce")

if "Installed power [MVA]" in dim_installation.columns:
    mva_to_kva = pd.to_numeric(dim_installation["Installed power [MVA]"], errors="coerce") * 1000
    dim_installation["Installed Power [kVA]"] = dim_installation["Installed Power [kVA]"].combine_first(mva_to_kva)

# --- 4. Rename columns ---
dim_installation = dim_installation.rename(columns={
    "Installation Code": "Installation_Code",
    "Type of installation": "Type_of_Installation",
    "Construction Type": "Construction_Type",
    "Usage Level [%]": "Usage_Level_Band",
    "Installed Power [kVA]": "Installed_Power_kVA",
    "Contracted Power [kVA]": "Contracted_Power_kVA",
    "Number of Clients": "Number_of_Clients",
    "Power Generation [kW]": "Power_Generation_kW",
    "Number of Client Producers": "Number_of_Client_Producers",
    "AT/MT transformation ratio [kV]": "at_mt_transformation_ratio_kv",
    "Tip [MW]": "tip_mv",
    "Maximum  short-circuit power HV [MVA]": "max_sc_power_hv_mv", 
    "Maximum short-circuit power MV [MVA]": "max_sc_power_mv_mv",
    "Minimum short-circuit power HV [MVA]": "min_sc_power_hv_mv",
    "Minimum short-circuit power MV [MVA]": "min_sc_power_mv_mv",
    "Short-circuit current for HV equipment sizing purposes [kA]": "sc_current_hv_ka"
})

# --- 5. Generate Surrogate Key ---
dim_installation = dim_installation.drop_duplicates(subset=["Installation_Code", "Busbar"]).reset_index(drop=True)
dim_installation["Installation_Id"] = dim_installation.index + 1

# --- 6. Finalize Table Structure ---
final_columns = [
    "Installation_Id",  "Installation_Code", "Designation", "Type_of_Installation",  "Busbar",  "Construction_Type",
    "Usage_Level_Band",  "Installed_Power_kVA",  "Contracted_Power_kVA",  "Number_of_Clients", "Power_Generation_kW",
    "Number_of_Client_Producers", "Latitude", "Longitude",  "at_mt_transformation_ratio_kv", "tip_mv",
    "max_sc_power_hv_mv", "max_sc_power_mv_mv", "min_sc_power_hv_mv",  "min_sc_power_mv_mv",   "sc_current_hv_ka",
]

dim_installation = dim_installation[final_columns]

print(dim_installation.head())

   Installation_Id Installation_Code         Designation Type_of_Installation  \
0                1      0105S5003900            Esgueira           Subestação   
1                2      0107S5002500             Espinho           Subestação   
2                3      0108S5001000              Avanca           Subestação   
3                4      0108S5001000              Avanca           Subestação   
4                5      0114S5022700  Oliveira do Bairro           Subestação   

          Busbar Construction_Type Usage_Level_Band  Installed_Power_kVA  \
0  15 KV - I/III               NaN              NaN              63000.0   
1     15 KV - I1               NaN              NaN              63000.0   
2     15 KV - I1               NaN              NaN              51500.0   
3     15 KV - I2               NaN              NaN              51500.0   
4          15 KV               NaN              NaN              31500.0   

   Contracted_Power_kVA  Number_of_Clients  ...  Number_

In [46]:
# ===========================================
# ETL: BUILD DIM_DATE
# ===========================================

# ---- 1. Collect all dates from all sheets ----
date_cols = ["Start Date", "End Date"]

dates = pd.concat([
    dips[date_cols].stack(),
    overs[date_cols].stack(),
    cont[date_cols].stack()
], ignore_index=True)

# Convert to datetime
dates = pd.to_datetime(dates, format="%m/%d/%Y", errors='coerce').dropna()

# Date data filtering
dates = dates[(dates.dt.year >= 2010) & (dates.dt.year <= 2025)]
dates = dates.drop_duplicates().sort_values().reset_index(drop=True)

# ---- 2. Build Dim_Date DataFrame ----
dim_date = pd.DataFrame({"Full_Date": dates})

# Primary Surrogate Key
dim_date["Date_Id"] = dim_date.index + 1

# Standard Temporal Attributes
dim_date["Year"] = dim_date["Full_Date"].dt.year
dim_date["Quarter"] = dim_date["Full_Date"].dt.quarter
dim_date["Month"] = dim_date["Full_Date"].dt.month
dim_date["Month_Name"] = dim_date["Full_Date"].dt.strftime("%B")
dim_date["Day"] = dim_date["Full_Date"].dt.day
dim_date["Week_Number"] = dim_date["Full_Date"].dt.isocalendar().week
dim_date["Day_of_Week"] = dim_date["Full_Date"].dt.day_name()
dim_date["Is_Weekend"] = dim_date["Day_of_Week"].isin(["Saturday", "Sunday"])

# ---- 3. Generate Level Keys (_Id) ----
dim_date["Year_Id"] = pd.factorize(dim_date["Year"])[0] + 1

# Quarter_Id: Unique integer for each specific Year-Quarter combo (e.g., 2014-Q1)
quarter_combo = dim_date["Year"].astype(str) + "-Q" + dim_date["Quarter"].astype(str)
dim_date["Quarter_Id"] = pd.factorize(quarter_combo)[0] + 1

# Month_Id: Unique integer for each specific Year-Month combo (e.g., 2014-01)
month_combo = dim_date["Full_Date"].dt.to_period("M")
dim_date["Month_Id"] = pd.factorize(month_combo)[0] + 1


# ---- 4. Reorder columns to match the DW spec image ----
final_columns = [ "Date_Id",  "Full_Date",  "Day",  "Month_Id",  "Month", "Month_Name", 
    "Quarter_Id",  "Quarter",  "Year_Id",  "Year", 
    "Week_Number",  "Day_of_Week", "Is_Weekend"]

dim_date = dim_date[final_columns]

# Display the final output
print(dim_date.head(n=5))

   Date_Id  Full_Date  Day  Month_Id  Month Month_Name  Quarter_Id  Quarter  \
0        1 2013-12-30   30         1     12   December           1        4   
1        2 2014-01-01    1         2      1    January           2        1   
2        3 2014-01-09    9         2      1    January           2        1   
3        4 2014-01-10   10         2      1    January           2        1   
4        5 2014-01-11   11         2      1    January           2        1   

   Year_Id  Year  Week_Number Day_of_Week  Is_Weekend  
0        1  2013            1      Monday       False  
1        2  2014            1   Wednesday       False  
2        2  2014            2    Thursday       False  
3        2  2014            2      Friday       False  
4        2  2014            2    Saturday        True  


In [47]:
# ===========================================
# ETL: BUILD DIM_LOCATION
# ===========================================

# Clean Text for Merging
def clean_for_merge(series):
    return (series.fillna("").astype(str).apply(lambda x: unicodedata.normalize('NFKD', x)
            .encode('ASCII', 'ignore').decode('utf-8')).str.replace('-', ' ', regex=False) 
            .str.replace(r'\xa0|\s+', ' ', regex=True).str.strip().str.upper())


# --- 1. Extract ---
municipality_subs = subs[["Municipality", "District", "DistrictCode", "DistrictMunicipalityCode"]].drop_duplicates()

municipality_events = pd.concat([
    dips[["Municipality", "NUTS III"]],
    overs[["Municipality", "NUTS III"]],
    cont[["Municipality", "NUTS III"]]
], ignore_index=True).drop_duplicates()

dim_location = pd.merge(municipality_subs, municipality_events, on="Municipality", how="left")

dim_location = dim_location.rename(columns={
    "DistrictMunicipalityCode": "Municipality_Code",
    "DistrictCode": "District_Code",
    "NUTS III": "NUTS_III"})

# --- 2. Transform ---

# Prepare nuts dataset
nuts_clean = nuts.rename(columns={
    "municipality_name": "Municipality",
    "municipality_code": "Municipality_Code",
    "nuts_3": "NUTS_III",
    "nuts_3_code": "NUTS_III_Code",
    "nuts_2": "NUTS_II",
    "nuts_2_code": "NUTS_II_Code"
}).drop_duplicates(subset=["Municipality"])

# Create safe merge keys for Municipalities
dim_location["Muni_Key"] = clean_for_merge(dim_location["Municipality"])
nuts_clean["Muni_Key"] = clean_for_merge(nuts_clean["Municipality"])

# --- 3. Full Join by Municipality ---
dim_location = pd.merge(
    dim_location,
    nuts_clean.drop(columns=["Municipality"]), 
    on="Muni_Key",
    how="left",
    suffixes=('_old', '_new'))

# --- 4. Resolve Missing values and Discrepancies ---

# Coalesce overlapping columns (Keep 'new' if it exists, otherwise fall back to 'old')
overlap_cols = ["Municipality_Code", "NUTS_III"] 
for col in overlap_cols:
    dim_location[col] = dim_location[f"{col}_new"].combine_first(dim_location[f"{col}_old"])
    dim_location = dim_location.drop(columns=[f"{col}_old", f"{col}_new"])

# Translate old NUTS III names to match official names
nuts_name_mapping = {
    "Alto Tâmega": "Alto Tâmega e Barroso",
    "Área Metropolitana do Porto": "Área Metropolitana do Porto", 
    "Área Metropolitana de Lisboa": "Grande Lisboa"}

dim_location["NUTS_III"] = dim_location["NUTS_III"].replace(nuts_name_mapping)

# Create a lookup table of the NUTS hierarchy 
nuts_hierarchy = nuts_clean[["NUTS_III", "NUTS_III_Code", "NUTS_II", "NUTS_II_Code"]].copy()
nuts_hierarchy["NUTS_Key"] = clean_for_merge(nuts_hierarchy["NUTS_III"])
nuts_hierarchy = nuts_hierarchy.dropna(subset=["NUTS_Key"]).drop_duplicates(subset=["NUTS_Key"])

# Create a matching NUTS key on main table
dim_location["NUTS_Key"] = clean_for_merge(dim_location["NUTS_III"])

# Merge to fetch the missing codes based on the NUTS name
dim_location = pd.merge(
    dim_location,
    nuts_hierarchy.drop(columns=["NUTS_III"]),
    on="NUTS_Key",
    how="left",
    suffixes=('', '_lookup')
)

# Fill the remaining NaN codes with the ones we just looked up
cols_to_fill = ["NUTS_III_Code", "NUTS_II", "NUTS_II_Code"]
for col in cols_to_fill:
    dim_location[col] = dim_location[col].combine_first(dim_location[f"{col}_lookup"])
    dim_location = dim_location.drop(columns=[f"{col}_lookup"])

# Cleanup temporary merge keys
dim_location = dim_location.drop(columns=["Muni_Key", "NUTS_Key"])

# --- 5. Deduplicate & Generate Keys ---
dim_location = dim_location.drop_duplicates(subset=["Municipality"]).reset_index(drop=True)

dim_location["Municipality_Id"] = dim_location.index + 1
dim_location["District_Id"] = pd.factorize(dim_location["District_Code"].fillna("UNKNOWN"))[0] + 1
dim_location["NUTS_III_Id"] = pd.factorize(dim_location["NUTS_III_Code"].fillna("UNKNOWN"))[0] + 1
dim_location["NUTS_II_Id"]  = pd.factorize(dim_location["NUTS_II_Code"].fillna("UNKNOWN"))[0] + 1

# --- 6. Arrange Columns ---
final_columns = [
    "Municipality_Id", "Municipality_Code", "Municipality",
    "District_Id", "District_Code", "District",
    "NUTS_III_Id", "NUTS_III_Code", "NUTS_III",
    "NUTS_II_Id", "NUTS_II_Code", "NUTS_II"
]

dim_location = dim_location[final_columns]

print(dim_location.head(n=5))

   Municipality_Id Municipality_Code          Municipality  District_Id  \
0                1              1703                Chaves            1   
1                2               109  Santa Maria da Feira            2   
2                3            182341                 Viseu            3   
3                4            110707                Loures            4   
4                5              1115               Amadora            4   

   District_Code   District  NUTS_III_Id NUTS_III_Code  \
0             17  Vila Real            1         PT11B   
1              1     Aveiro            2         PT11A   
2             18      Viseu            3         PT194   
3             11     Lisboa            4         PT1A0   
4             11     Lisboa            4         PT1A0   

                      NUTS_III  NUTS_II_Id NUTS_II_Code        NUTS_II  
0        Alto Tâmega e Barroso           1         PT11          Norte  
1  Área Metropolitana do Porto           1         PT1

In [ ]:
# ===========================================
# ETL: BUILD DIM_VOLTAGE_LEVEL
# ===========================================

# --- 1. Collect all voltages from event sheets ---
all_voltages = pd.concat([
    dips["Nominal Voltage (V)"],
    overs["Nominal Voltage (V)"],
    cont["Nominal Voltage (V)"]
], ignore_index=True)

# Drop nulls, remove duplicates, sort, and reset index
all_voltages = all_voltages.dropna().drop_duplicates().sort_values().reset_index(drop=True)

dim_voltage_level = pd.DataFrame({"Nominal_Voltage_V": all_voltages})

# --- 2. Define Category Functions ---
def voltage_band(volts):
    if pd.isna(volts) or volts == 0:
        return "NONE"
    elif volts <= 1000:
        return "LV"
    elif volts <= 45000:
        return "MV"
    elif volts <= 110000:
        return "HV"
    else:
        return "EHV"

def voltage_desc(volts):
    if pd.isna(volts) or volts == 0:
        return "No Voltage"
    elif volts <= 1000:
        return "Low Voltage"
    elif volts <= 45000:
        return "Medium Voltage"
    elif volts <= 110000:
        return "High Voltage"
    else:
        return "Extra High Voltage"

# --- 3. Apply Transformations ---
dim_voltage_level["Voltage_Band_Code"] = dim_voltage_level["Nominal_Voltage_V"].apply(voltage_band)
dim_voltage_level["Voltage_Band_Description"] = dim_voltage_level["Nominal_Voltage_V"].apply(voltage_desc)

# Generate Surrogate Key (Starting at 1)
dim_voltage_level["Voltage_Level_Id"] = dim_voltage_level.index + 1

# --- 4. Add the 'Unknown' Row (ID 0) ---
unknown_row = pd.DataFrame([{
    "Voltage_Level_Id": 0,
    "Nominal_Voltage_V": 0,
    "Voltage_Band_Code": "UNKNOWN",
    "Voltage_Band_Description": "Unknown / Unmapped"
}])

# Concatenate the unknown row to the top of the dimension table
dim_voltage_level = pd.concat([unknown_row, dim_voltage_level], ignore_index=True)

# --- 5. Reorder Columns to match DW Spec ---
dim_voltage_level = dim_voltage_level[[
    "Voltage_Level_Id",
    "Nominal_Voltage_V",
    "Voltage_Band_Code",
    "Voltage_Band_Description"
]]

# Display the final output
print(dim_voltage_level.head(n=5))

   Voltage_Level_Id  Nominal_Voltage_V Voltage_Band_Code  \
0                 0                0.0           UNKNOWN   
1                 1              230.0                LV   
2                 2             6000.0                MV   
3                 3            10000.0                MV   
4                 4            15000.0                MV   

  Voltage_Band_Description  
0       Unknown / Unmapped  
1              Low Voltage  
2           Medium Voltage  
3           Medium Voltage  
4           Medium Voltage  


In [49]:
# ===========================================
# ETL: BUILD DIM_VOLTAGE_CLASS (E-REDES / EN 50160 Standard)
# ===========================================

# 1. Official EN 50160 / ERSE Mapping Dictionaries
# Residual Voltage Ranges (Dips)
dip_voltages = {
    "A": {"min": 80, "max": 90, "desc": "80% to 90%"},
    "B": {"min": 70, "max": 80, "desc": "70% to 80%"},
    "C": {"min": 40, "max": 70, "desc": "40% to 70%"},
    "D": {"min": 5,  "max": 40, "desc": "5% to 40%"},
    "X": {"min": 0,  "max": 5,  "desc": "<5% (Interruption)"}}

# Incremental Voltage Ranges (Overvoltages)
over_voltages = {
    "T": {"min": 110, "max": 119, "desc": "110% to 120%"},
    "S": {"min": 120, "max": 9999999, "desc": ">120%"}} # Unbounded max

# Standardized Durations (Dips)
durations_dip_voltages = {       
    "1": {"min": 10,   "max": 200,   "desc": "10ms to 200ms"},
    "2": {"min": 201,  "max": 500,   "desc": "200ms to 500ms"},
    "3": {"min": 501,  "max": 1000,  "desc": "500ms to 1s"},
    "4": {"min": 1001, "max": 5000,  "desc": "1s to 5s"},
    "5": {"min": 5001, "max": 60000, "desc": "5s to 1min"}}

# Standardized Durations (Overvoltages)
durations_over_voltages = {       
    "1": {"min": 10,   "max": 500,   "desc": "10ms to 500ms"},
    "2": {"min": 501,  "max": 5000,   "desc": "500ms to 5s"},
    "3": {"min": 5001, "max": 60000, "desc": "5s to 1min"}}

# --- 2. Define Severity Risk Mapping ---
# Maps Voltage_Class_Code to (Severity_Risk_Level, Severity_Risk_Label)
risk_mapping = {
    # DIPS (Sags)
    'A1': (1, 'Low Risk'), 'A2': (1, 'Low Risk'), 'A3': (1, 'Low Risk'), 
    'B1': (1, 'Low Risk'), 'B2': (1, 'Low Risk'), 'C1': (1, 'Low Risk'),
    'A4': (2, 'Moderate Risk'), 'B3': (2, 'Moderate Risk'), 'C2': (2, 'Moderate Risk'),
    'A5': (3, 'High Risk'), 'B4': (3, 'High Risk'), 'C3': (3, 'High Risk'), 
    'C4': (3, 'High Risk'), 'D1': (3, 'High Risk'), 'D2': (3, 'High Risk'), 'X1': (3, 'High Risk'),
    'B5': (4, 'Critical Risk'), 'C5': (4, 'Critical Risk'), 'D3': (4, 'Critical Risk'), 
    'D4': (4, 'Critical Risk'), 'D5': (4, 'Critical Risk'), 'X2': (4, 'Critical Risk'), 
    'X3': (4, 'Critical Risk'), 'X4': (4, 'Critical Risk'), 'X5': (4, 'Critical Risk'),
    
    # OVERVOLTAGES (Swells)
    'T1': (2, 'Moderate Risk'),
    'T2': (3, 'High Risk'), 'S1': (3, 'High Risk'),
    'T3': (4, 'Critical Risk'), 'S2': (4, 'Critical Risk'),
    'S3': (5, 'Extreme Risk')}

class_data = []

# --- 3. Generate DIPS (A1 to X5) ---
for family in dip_voltages.keys():
    for duration in ["1", "2", "3", "4", "5"]:
        code = f"{family}{duration}"
        risk_level, risk_label = risk_mapping.get(code, (0, "Unknown"))
        
        class_data.append({
            "Voltage_Class_Code": code,
            "Voltage_Event_Type": "Voltage Dip",
            "Magnitude_Basis": "Residual Voltage",
            "Class_Family_Code": family,
            "Duration_Class_Code": duration,
            "Voltage_Range_Description": dip_voltages[family]["desc"],
            "Duration_Range_Description": durations_dip_voltages[duration]["desc"],
            "Voltage_Min_Pct": dip_voltages[family]["min"],
            "Voltage_Max_Pct": dip_voltages[family]["max"],
            "Duration_Min_ms": durations_dip_voltages[duration]["min"],
            "Duration_Max_ms": durations_dip_voltages[duration]["max"],
            "Severity_Rank": risk_level})

# --- 4. Generate OVERVOLTAGES (S1 to T3) ---
for family in over_voltages.keys():
    for duration in ["1", "2", "3"]: 
        code = f"{family}{duration}"
        risk_level, risk_label = risk_mapping.get(code, (0, "Unknown"))
        
        class_data.append({
            "Voltage_Class_Code": code,
            "Voltage_Event_Type": "Overvoltage",
            "Magnitude_Basis": "Incremental Voltage",
            "Class_Family_Code": family,
            "Duration_Class_Code": duration,
            "Voltage_Range_Description": over_voltages[family]["desc"],
            "Duration_Range_Description": durations_over_voltages[duration]["desc"],
            "Voltage_Min_Pct": over_voltages[family]["min"],
            "Voltage_Max_Pct": over_voltages[family]["max"],
            "Duration_Min_ms": durations_over_voltages[duration]["min"],
            "Duration_Max_ms": durations_over_voltages[duration]["max"],
            "Severity_Rank": risk_level})

# --- 5. Finalize DataFrame ---
dim_voltage_class = pd.DataFrame(class_data)
dim_voltage_class.insert(0, "Voltage_Class_Id", dim_voltage_class.index + 1)

final_columns = [
    "Voltage_Class_Id", "Voltage_Class_Code", "Voltage_Event_Type",
    "Magnitude_Basis", "Class_Family_Code", "Duration_Class_Code",
    "Voltage_Range_Description", "Duration_Range_Description",
    "Voltage_Min_Pct", "Voltage_Max_Pct", "Duration_Min_ms",
    "Duration_Max_ms", "Severity_Rank"]

dim_voltage_class = dim_voltage_class[final_columns]

print(dim_voltage_class.head(n=5))

   Voltage_Class_Id Voltage_Class_Code Voltage_Event_Type   Magnitude_Basis  \
0                 1                 A1        Voltage Dip  Residual Voltage   
1                 2                 A2        Voltage Dip  Residual Voltage   
2                 3                 A3        Voltage Dip  Residual Voltage   
3                 4                 A4        Voltage Dip  Residual Voltage   
4                 5                 A5        Voltage Dip  Residual Voltage   

  Class_Family_Code Duration_Class_Code Voltage_Range_Description  \
0                 A                   1                80% to 90%   
1                 A                   2                80% to 90%   
2                 A                   3                80% to 90%   
3                 A                   4                80% to 90%   
4                 A                   5                80% to 90%   

  Duration_Range_Description  Voltage_Min_Pct  Voltage_Max_Pct  \
0              10ms to 200ms               8

In [50]:
# ===========================================
# ETL: BUILD DIM_EXCEPTIONAL_EVENT
# ===========================================

# --- 1. Collect events from dips and overs ---
events_discrete = pd.concat([
    dips["Exceptional Event"],
    overs["Exceptional Event"]
], ignore_index=True)

# --- 2. Collect events from continuous phenomena ---
events_continuous = cont["Observations"] 

# --- 3. Combine all sources ---
all_events = pd.concat([events_discrete, events_continuous], ignore_index=True)

# --- 4. Clean the text data ---
# Drop true nulls, strip white spaces to prevent accidental duplicates
all_events = all_events.dropna().str.strip()

# Remove any empty strings that might remain after stripping
all_events = all_events[all_events != ""]

correcoes_eventos = {
    "Evento Excecional - Depressão 1 e 2 março": "Evento Excecional - Depressão 1 e 2 de março",
    "Incidente de Grande Impacto - Incêndios de outubro (em processo de classificação como evento excecional)": "Evento Excecional - Incêndios de outubro",
    "Incidente de Grande Impacto - Incêndios de junho                                             (em processo de classificação como evento excecional)": "Evento Excecional - Incêndios de junho"}

# Apply corrections
all_events = all_events.replace(correcoes_eventos)

# Drop duplicates, sort alphabetically, and reset index
all_events = all_events.drop_duplicates().sort_values().reset_index(drop=True)

# --- 5. Build the DataFrame ---
dim_exceptional_event = pd.DataFrame({"Exceptional_Event_Name": all_events})

# Generate Surrogate Key (Starting at 1)
dim_exceptional_event["Exceptional_Event_Id"] = dim_exceptional_event.index + 1

# --- 6. Add a default "No Event" row (Highly Recommended for DW) ---
no_event_row = pd.DataFrame({
    "Exceptional_Event_Id": [0], 
    "Exceptional_Event_Name": ["No Exceptional Event / Normal Conditions"]})

dim_exceptional_event = pd.concat([no_event_row, dim_exceptional_event], ignore_index=True)


# --- 7. Reorder columns to match the DW spec ---
dim_exceptional_event = dim_exceptional_event[["Exceptional_Event_Id", "Exceptional_Event_Name"]]

# Display the final output
print(dim_exceptional_event.head(n=5))

   Exceptional_Event_Id                        Exceptional_Event_Name
0                     0      No Exceptional Event / Normal Conditions
1                     1              Evento Excecional - Ciclone Kirk
2                     2  Evento Excecional - Depressão 1 e 2 de março
3                     3           Evento Excecional - Depressão Aline
4                     4          Evento Excecional - Depressão Ciarán


# Export Dimensions

In [51]:
# ===========================================
# EXPORT ALL DIMENSIONS TO ONE EXCEL FILE
# ===========================================

with pd.ExcelWriter("DIMENSIONS_OUTPUT.xlsx", engine="openpyxl") as writer:
    dim_date.to_excel(writer, sheet_name="Dim_Date", index=False)
    dim_installation.to_excel(writer, sheet_name="Dim_Installation", index=False)
    dim_location.to_excel(writer, sheet_name="Dim_Location", index=False)
    dim_voltage_level.to_excel(writer, sheet_name="Dim_Voltage_Level", index=False)
    dim_voltage_class.to_excel(writer, sheet_name="Dim_Voltage_Class", index=False)
    dim_exceptional_event.to_excel(writer, sheet_name="Dim_Exceptional_Event", index=False)

print("Dimensions exported to DIMENSIONS_OUTPUT.xlsx")

Dimensions exported to DIMENSIONS_OUTPUT.xlsx


# Create Fact Tables

In [52]:
# ===========================================
# ETL: BUILD FACT_VOLTAGE_EVENT
# ===========================================

# --- 1. Identify columns  ---
dip_classes = dim_voltage_class[dim_voltage_class['Voltage_Event_Type'] == 'Voltage Dip']['Voltage_Class_Code'].tolist()
over_classes = dim_voltage_class[dim_voltage_class['Voltage_Event_Type'] == 'Overvoltage']['Voltage_Class_Code'].tolist()

dip_cols_to_melt = [c for c in dip_classes if c in dips.columns]
over_cols_to_melt = [c for c in over_classes if c in overs.columns]

id_vars_cols = ["Installation Code", "Busbar", "Nominal Voltage (V)", "Start Date", "End Date", "Exceptional Event", "Municipality"]

# --- 2. Melt and Combine ---
dips_melted = pd.melt(dips, id_vars=id_vars_cols, value_vars=dip_cols_to_melt, var_name="Voltage_Class_Code", value_name="event_count")
overs_melted = pd.melt(overs, id_vars=id_vars_cols, value_vars=over_cols_to_melt, var_name="Voltage_Class_Code", value_name="event_count")
fact_events_raw = pd.concat([dips_melted, overs_melted], ignore_index=True)
fact_events_raw = fact_events_raw[fact_events_raw["event_count"] > 0].copy()

# --- 3. Clean string data ---
fact_events_raw["Exceptional Event"] = fact_events_raw["Exceptional Event"].str.strip().replace(correcoes_eventos)
fact_events_raw["Exceptional Event"] = fact_events_raw["Exceptional Event"].fillna("No Exceptional Event / Normal Conditions")

fact_events_raw["Municipality_Clean"] = clean_for_merge(fact_events_raw["Municipality"])
dim_location_subset = dim_location[['Municipality', 'Municipality_Id']].copy()
dim_location_subset['Muni_Match_Key'] = clean_for_merge(dim_location_subset['Municipality'])

# --- 4. Merge to get Surrogate Keys ---
fact_events_raw["Start Date"] = pd.to_datetime(fact_events_raw["Start Date"])
fact_events_raw["End Date"] = pd.to_datetime(fact_events_raw["End Date"])

# Merge datasets
fact_events = pd.merge(fact_events_raw, dim_date[['Full_Date', 'Date_Id']], left_on="Start Date", right_on="Full_Date", how="left").rename(columns={"Date_Id": "start_date_id"}).drop(columns=["Full_Date"])
fact_events = pd.merge(fact_events, dim_date[['Full_Date', 'Date_Id']], left_on="End Date", right_on="Full_Date", how="left").rename(columns={"Date_Id": "end_date_id"}).drop(columns=["Full_Date"])

fact_events = pd.merge(fact_events, dim_installation[['Installation_Code', 'Busbar', 'Installation_Id']], left_on=["Installation Code", "Busbar"], right_on=["Installation_Code", "Busbar"], how="left").rename(columns={"Installation_Id": "installation_id"})
fact_events = pd.merge(fact_events, dim_voltage_level[['Nominal_Voltage_V', 'Voltage_Level_Id']], left_on="Nominal Voltage (V)", right_on="Nominal_Voltage_V", how="left").rename(columns={"Voltage_Level_Id": "voltage_level_id"})
fact_events = pd.merge(fact_events, dim_voltage_class[['Voltage_Class_Code', 'Voltage_Class_Id']], on="Voltage_Class_Code", how="left").rename(columns={"Voltage_Class_Id": "voltage_class_id"})
fact_events = pd.merge(fact_events, dim_exceptional_event[['Exceptional_Event_Name', 'Exceptional_Event_Id']], left_on="Exceptional Event", right_on="Exceptional_Event_Name", how="left").rename(columns={"Exceptional_Event_Id": "exceptional_event_id"})

fact_events = pd.merge(
    fact_events, 
    dim_location_subset[['Muni_Match_Key', 'Municipality_Id']], 
    left_on="Municipality_Clean", 
    right_on="Muni_Match_Key", 
    how="left").rename(columns={"Municipality_Id": "municipality_id"})

# --- 5. Add base measures ---
fact_events["record_count"] = 1

# Unknow or unmapped IDs get default 0
cols_to_fill = ["installation_id", "voltage_level_id", "municipality_id", "exceptional_event_id", "start_date_id", "end_date_id", "voltage_class_id"]
for col in cols_to_fill:
    fact_events[col] = fact_events[col].fillna(0).astype(int)
fact_events["exceptional_flag"] = (fact_events["exceptional_event_id"] > 0).astype(int)

# --- 6. Finalize Fact Table ---
fact_voltage_event = fact_events[[
    "event_count", "exceptional_flag", "record_count", 
    "start_date_id", "end_date_id", "installation_id", 
    "voltage_level_id", "voltage_class_id", "exceptional_event_id",
    "municipality_id"]].copy()

In [53]:
# ===========================================
# ETL: BUILD FACT_CONTINUOUS_PHENOMENA
# ===========================================

# --- 1. Initial Selection and Cleaning ---
fact_cont_raw = cont.copy()

# Clean the Municipality names for matching
fact_cont_raw["Municipality_Clean"] = clean_for_merge(fact_cont_raw["Municipality"])

# Prepare dim_location subset for matching 
dim_location_subset = dim_location[['Municipality', 'Municipality_Id']].copy()
dim_location_subset['Muni_Match_Key'] = clean_for_merge(dim_location_subset['Municipality'])

# Calculate Monitoring Days
fact_cont_raw["Start Date"] = pd.to_datetime(fact_cont_raw["Start Date"])
fact_cont_raw["End Date"] = pd.to_datetime(fact_cont_raw["End Date"])
fact_cont_raw["monitoring_days"] = (fact_cont_raw["End Date"] - fact_cont_raw["Start Date"]).dt.days

# --- 2 Map exceptional event from Observations ---
fact_cont_raw["Observations"] = fact_cont_raw["Observations"].str.strip().replace(correcoes_eventos)
fact_cont_raw["Observations"] = fact_cont_raw["Observations"].fillna("No Exceptional Event / Normal Conditions")

# --- 3. Merge ---

# Merge datasets to get all surrogate keys
fact_cont = pd.merge(fact_cont_raw, dim_date[['Full_Date', 'Date_Id']], left_on="Start Date", right_on="Full_Date", how="left").rename(columns={"Date_Id": "start_date_id"}).drop(columns=["Full_Date"])
fact_cont = pd.merge(fact_cont, dim_date[['Full_Date', 'Date_Id']], left_on="End Date", right_on="Full_Date", how="left").rename(columns={"Date_Id": "end_date_id"}).drop(columns=["Full_Date"])
fact_cont = pd.merge(fact_cont, dim_installation[['Installation_Code', 'Busbar', 'Installation_Id']], left_on=["Installation Code", "Busbar"], right_on=["Installation_Code", "Busbar"], how="left").rename(columns={"Installation_Id": "installation_id"})
fact_cont = pd.merge(fact_cont, dim_voltage_level[['Nominal_Voltage_V', 'Voltage_Level_Id']], left_on="Nominal Voltage (V)", right_on="Nominal_Voltage_V", how="left").rename(columns={"Voltage_Level_Id": "voltage_level_id"})
fact_cont = pd.merge(fact_cont, dim_exceptional_event[['Exceptional_Event_Name', 'Exceptional_Event_Id']], left_on="Observations", right_on="Exceptional_Event_Name", how="left").rename(columns={"Exceptional_Event_Id": "exceptional_event_id"})

# Merge to get Municipality_Id 
fact_cont = pd.merge(
    fact_cont, 
    dim_location_subset[['Muni_Match_Key', 'Municipality_Id']], 
    left_on="Municipality_Clean", 
    right_on="Muni_Match_Key", 
    how="left"
).rename(columns={"Municipality_Id": "municipality_id"})

# Fill unmapped IDs with 0 (prevents data loss)
cols_to_fill_cont = [
    "exceptional_event_id", "installation_id", "voltage_level_id", 
    "municipality_id", "start_date_id", "end_date_id"]
for col in cols_to_fill_cont:
    fact_cont[col] = fact_cont[col].fillna(0)

fact_cont["voltage_class_id"] = -1 

# --- 4 Rename columns ---
fact_cont = fact_cont.rename(columns={
    "Voltage L1": "voltage_l1_pct", "Voltage L2": "voltage_l2_pct", "Voltage L3": "voltage_l3_pct",
    "Flicker L1": "flicker_l1_pct", "Flicker L2": "flicker_l2_pct", "Flicker L3": "flicker_l3_pct",
    "THD L1": "thd_l1_pct", "THD L2": "thd_l2_pct", "THD L3": "thd_l3_pct",
    "Unbalanced": "unbalanced_pct", "Frequency": "frequency_pct"})

# --- 5. Calculate norm_compliance ---
pct_cols = [
    "voltage_l1_pct", "voltage_l2_pct", "voltage_l3_pct",
    "flicker_l1_pct", "flicker_l2_pct", "flicker_l3_pct",
    "thd_l1_pct", "thd_l2_pct", "thd_l3_pct",
    "unbalanced_pct", "frequency_pct"]
for col in pct_cols:
    fact_cont[col] = pd.to_numeric(fact_cont[col], errors='coerce').fillna(0)
    fact_cont[col] = fact_cont[col].clip(lower=0, upper=100)
    
fact_cont["norm_compliance"] = fact_cont[pct_cols].mean(axis=1)

# --- 6. Finalize the fact table structure ---
fact_continuous_phenomena = fact_cont[[
    "voltage_l1_pct", "voltage_l2_pct", "voltage_l3_pct",
    "flicker_l1_pct", "flicker_l2_pct", "flicker_l3_pct",
    "thd_l1_pct", "thd_l2_pct", "thd_l3_pct",
    "unbalanced_pct", "frequency_pct",
    "monitoring_days", "norm_compliance", 
    "start_date_id", "end_date_id", "installation_id", 
    "voltage_level_id", "voltage_class_id", "exceptional_event_id",
    "municipality_id"]].copy()

# Cast integer columns 
int_cols_cont = [
    "monitoring_days", "start_date_id", "end_date_id", 
    "installation_id", "voltage_level_id", "voltage_class_id", 
    "exceptional_event_id", "municipality_id"]
fact_continuous_phenomena[int_cols_cont] = fact_continuous_phenomena[int_cols_cont].astype(int)

In [54]:
# ===========================================
# ETL: BUILD AGG_MUNICIPALITY_VOLTAGE_EVENT
# ===========================================

# --- 1. Merge to get Month_Id from dim_date ---
agg_events_base = pd.merge(
    fact_voltage_event, 
    dim_date[['Date_Id', 'Month_Id']], 
    left_on="start_date_id", 
    right_on="Date_Id", 
    how="left").rename(columns={"Month_Id": "month_id"})

# --- 2. Handle missing Month_Ids ---

agg_events_base["month_id"] = agg_events_base["month_id"].fillna(0)

# --- 3. Aggregate metrics ---
# Grouping by IDs to create the periodic snapshot
agg_municipality_voltage_event = agg_events_base.groupby(
    ["month_id", "municipality_id", "voltage_level_id", "voltage_class_id"], as_index=False).agg(
    total_event_count=("event_count", "sum"),
    total_record_count=("record_count", "sum"),
    installation_count=("installation_id", "nunique"))

# --- 4. Finalize Columns ---
# unmapped records have default ID 0
agg_municipality_voltage_event = agg_municipality_voltage_event[[
    "total_event_count", "total_record_count", "installation_count", 
    "month_id", "municipality_id", "voltage_level_id", "voltage_class_id"]].copy()

# --- 5. Convert IDs to integers ---
int_cols_agg = ["month_id", "municipality_id", "voltage_level_id", "voltage_class_id"]
agg_municipality_voltage_event[int_cols_agg] = agg_municipality_voltage_event[int_cols_agg].astype(int)


In [55]:
# ===========================================
# ETL: BUILD AGG_MUNICIPALITY_CONTINUOUS_PHENOMENA
# ===========================================

# --- 1. Merge to get Month_Id from dim_date ---
agg_cont_base = pd.merge(
    fact_continuous_phenomena, 
    dim_date[['Date_Id', 'Month_Id']], 
    left_on="start_date_id", 
    right_on="Date_Id", 
    how="left"
).rename(columns={"Month_Id": "month_id"})

# --- 2. Handle missing Month_Ids ---
agg_cont_base["month_id"] = agg_cont_base["month_id"].fillna(0)

# --- 3. Aggregate metrics ---
agg_municipality_continuous_phenomena = agg_cont_base.groupby(
    ["month_id", "municipality_id", "voltage_level_id"], as_index=False).agg(
    avg_voltage_l1_pct=("voltage_l1_pct", "mean"),
    avg_voltage_l2_pct=("voltage_l2_pct", "mean"),
    avg_voltage_l3_pct=("voltage_l3_pct", "mean"),
    avg_flicker_l1_pct=("flicker_l1_pct", "mean"),
    avg_flicker_l2_pct=("flicker_l2_pct", "mean"),
    avg_flicker_l3_pct=("flicker_l3_pct", "mean"),
    avg_thd_l1_pct=("thd_l1_pct", "mean"),
    avg_thd_l2_pct=("thd_l2_pct", "mean"),
    avg_thd_l3_pct=("thd_l3_pct", "mean"),
    avg_unbalanced_pct=("unbalanced_pct", "mean"),
    avg_frequency_pct=("frequency_pct", "mean"),
    total_monitoring_days=("monitoring_days", "sum"),
    installation_count=("installation_id", "nunique"))

# --- 4. Reorder columns and Finalize ---
# unmapped records have default ID 0
agg_municipality_continuous_phenomena = agg_municipality_continuous_phenomena[[
    "avg_voltage_l1_pct", "avg_voltage_l2_pct", "avg_voltage_l3_pct", 
    "avg_flicker_l1_pct", "avg_flicker_l2_pct", "avg_flicker_l3_pct", 
    "avg_thd_l1_pct", "avg_thd_l2_pct", "avg_thd_l3_pct",
    "avg_unbalanced_pct", "avg_frequency_pct", "total_monitoring_days",
    "installation_count", "month_id", "municipality_id", "voltage_level_id"]].copy()

# --- 5. Convert IDs to integers ---
int_cols_agg_cont = ["month_id", "municipality_id", "voltage_level_id"]
agg_municipality_continuous_phenomena[int_cols_agg_cont] = agg_municipality_continuous_phenomena[int_cols_agg_cont].astype(int)

# Inspect Fact Table Structure

In [56]:
fact_tables = {
    "fact_voltage_event": fact_voltage_event,
    "fact_continuous_phenomena": fact_continuous_phenomena,
    "agg_municipality_voltage_event": agg_municipality_voltage_event,
    "agg_municipality_continuous_phenomena": agg_municipality_continuous_phenomena}

for name, df in fact_tables.items():
    print(f"\n{'='*60}")
    print(f"TABLE: {name}")
    print(f"{'='*60}")
    
    print("COLUMNS:")
    print(df.columns.tolist())
    
    print("\nFIRST 5 ROWS:")
    display(df.head())


TABLE: fact_voltage_event
COLUMNS:
['event_count', 'exceptional_flag', 'record_count', 'start_date_id', 'end_date_id', 'installation_id', 'voltage_level_id', 'voltage_class_id', 'exceptional_event_id', 'municipality_id']

FIRST 5 ROWS:


,event_count,exceptional_flag,record_count,start_date_id,end_date_id,installation_id,voltage_level_id,voltage_class_id,exceptional_event_id,municipality_id
0,45,0,1,1,40,1,4,1,0,140
1,24,0,1,4,40,2,4,1,0,221
2,29,0,1,12,40,3,4,1,0,116
3,39,0,1,12,40,4,4,1,0,116
4,32,0,1,1,40,5,4,1,0,105



TABLE: fact_continuous_phenomena
COLUMNS:
['voltage_l1_pct', 'voltage_l2_pct', 'voltage_l3_pct', 'flicker_l1_pct', 'flicker_l2_pct', 'flicker_l3_pct', 'thd_l1_pct', 'thd_l2_pct', 'thd_l3_pct', 'unbalanced_pct', 'frequency_pct', 'monitoring_days', 'norm_compliance', 'start_date_id', 'end_date_id', 'installation_id', 'voltage_level_id', 'voltage_class_id', 'exceptional_event_id', 'municipality_id']

FIRST 5 ROWS:


,voltage_l1_pct,voltage_l2_pct,voltage_l3_pct,flicker_l1_pct,flicker_l2_pct,flicker_l3_pct,thd_l1_pct,thd_l2_pct,thd_l3_pct,unbalanced_pct,frequency_pct,monitoring_days,norm_compliance,start_date_id,end_date_id,installation_id,voltage_level_id,voltage_class_id,exceptional_event_id,municipality_id
0,100.0,100.0,100.0,99.91790,99.92990,99.92390,100,100,100,100.0,99.9981,359,99.979073,4,40,103,4,-1,0,238
1,100.0,100.0,100.0,99.96036,99.95243,100.00000,100,100,100,100.0,100.0000,363,99.992072,41,81,66,4,-1,0,61
2,100.0,100.0,100.0,99.89799,99.86185,99.89799,100,100,100,100.0,100.0000,335,99.968894,124,177,451,4,-1,0,83
3,100.0,100.0,100.0,99.95053,99.96908,99.94847,100,100,100,100.0,100.0000,363,99.988007,184,229,73,7,-1,0,248
4,100.0,100.0,100.0,99.71050,99.59800,99.80700,100,100,100,100.0,100.0000,86,99.919591,65,71,661,1,-1,0,103



TABLE: agg_municipality_voltage_event
COLUMNS:
['total_event_count', 'total_record_count', 'installation_count', 'month_id', 'municipality_id', 'voltage_level_id', 'voltage_class_id']

FIRST 5 ROWS:


,total_event_count,total_record_count,installation_count,month_id,municipality_id,voltage_level_id,voltage_class_id
0,104,3,3,1,3,4,1
1,12,3,3,1,3,4,2
2,6,2,2,1,3,4,3
3,4,3,3,1,3,4,4
4,1,1,1,1,3,4,5



TABLE: agg_municipality_continuous_phenomena
COLUMNS:
['avg_voltage_l1_pct', 'avg_voltage_l2_pct', 'avg_voltage_l3_pct', 'avg_flicker_l1_pct', 'avg_flicker_l2_pct', 'avg_flicker_l3_pct', 'avg_thd_l1_pct', 'avg_thd_l2_pct', 'avg_thd_l3_pct', 'avg_unbalanced_pct', 'avg_frequency_pct', 'total_monitoring_days', 'installation_count', 'month_id', 'municipality_id', 'voltage_level_id']

FIRST 5 ROWS:


,avg_voltage_l1_pct,avg_voltage_l2_pct,avg_voltage_l3_pct,avg_flicker_l1_pct,avg_flicker_l2_pct,avg_flicker_l3_pct,avg_thd_l1_pct,avg_thd_l2_pct,avg_thd_l3_pct,avg_unbalanced_pct,avg_frequency_pct,total_monitoring_days,installation_count,month_id,municipality_id,voltage_level_id
0,100.0,100.0,100.0,99.910733,99.879200,99.876667,100.0,100.0,100.0,100.0,100.0,1110,3,1,3,4
1,100.0,100.0,100.0,99.945100,99.937500,99.922300,100.0,100.0,100.0,100.0,100.0,370,1,1,4,3
2,100.0,100.0,100.0,99.973500,99.926200,99.947000,100.0,100.0,100.0,100.0,100.0,370,1,1,5,3
3,100.0,100.0,100.0,99.971700,99.953786,99.966643,100.0,100.0,100.0,100.0,100.0,2590,7,1,7,3
4,100.0,100.0,100.0,99.970600,99.974500,99.974500,100.0,100.0,100.0,100.0,100.0,370,1,1,12,7


# Export Fact Tables to Excel

In [ ]:
# ==============================================================================
# EXPORT
# ==============================================================================
with pd.ExcelWriter("FACTS_OUTPUT.xlsx", engine="openpyxl") as writer:
    fact_voltage_event.to_excel(writer, sheet_name="fact_voltage_event", index=False)
    fact_continuous_phenomena.to_excel(writer, sheet_name="fact_continuous_phen", index=False)
    agg_municipality_voltage_event.to_excel(writer, sheet_name="agg_muni_voltage", index=False)
    agg_municipality_continuous_phenomena.to_excel(writer, sheet_name="agg_muni_cont", index=False)
print("Fact tables exported to FACTS_OUTPUT.xlsx")